In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/vivek468/superstore-dataset-final/Sample - Superstore.csv


## Load The Dataset

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import warnings
warnings.filterwarnings("ignore")

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
df = pd.read_csv('/kaggle/input/datasets/vivek468/superstore-dataset-final/Sample - Superstore.csv',
                  encoding='latin1')

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
df.head()

Shape: (9994, 21)

Columns: ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


**Clean, Fix Date columns and Explore Basic Statistics**

In [4]:
print("Data Types\n:", df.dtypes)
print("Missing Values\n:", df.isnull().sum())
print("Basic Statistics:")
print(df[["Sales", "Discount", "Quantity","Profit"]].describe())

Data Types
: Row ID             int64
Order ID          object
Order Date        object
Ship Date         object
Ship Mode         object
Customer ID       object
Customer Name     object
Segment           object
Country           object
City              object
State             object
Postal Code        int64
Region            object
Product ID        object
Category          object
Sub-Category      object
Product Name      object
Sales            float64
Quantity           int64
Discount         float64
Profit           float64
dtype: object
Missing Values
: Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64
Basic Statistics:
 

In [5]:
df["Order Date"] = pd.to_datetime(df["Order Date"])
df["Ship Date"] = pd.to_datetime(df["Ship Date"])
df["Year"] = df["Order Date"].dt.year
df["Month"] = df["Order Date"].dt.month
df["Month_Name"] = df["Order Date"].dt.strftime('%B')

print("Date columns fixed!")
print(df[["Order Date", "Year","Month", "Month_Name"]].head())

Date columns fixed!
  Order Date  Year  Month Month_Name
0 2016-11-08  2016     11   November
1 2016-11-08  2016     11   November
2 2016-06-12  2016      6       June
3 2015-10-11  2015     10    October
4 2015-10-11  2015     10    October


**Load Dataset into Sqlite3**

In [6]:
import sqlite3

conn = sqlite3.connect('superstore.db')

df.to_sql('superstore', conn, 
          if_exists='replace', 
          index=False)

print("Data loaded into SQLite successfully!")
print(f"Total rows: {pd.read_sql('SELECT COUNT(*) FROM superstore', conn).values[0][0]}")

Data loaded into SQLite successfully!
Total rows: 9994


In [7]:
def run_query(query, title=""):
    result = pd.read_sql(query, conn)
    if title:
        print(f"\n{'='*50}")
        print(f"  {title}")
        print(f"{'='*50}")
        return result

## Analysis

In [8]:
conn = sqlite3.connect('superstore.db')
run_query("""
SELECT 
ROUND(SUM(Sales), 2) AS TOTAL_REVENUE,
ROUND(SUM(Profit), 2) AS TOTAL_PROFIT,
ROUND(AVG(Profit/Sales)*100, 2) AS PROFIT_MARGIN_PCT,
COUNT(DISTINCT "Order ID") AS TOTAL_ORDERS,
COUNT(DISTINCT "Customer ID") AS TOTAL_CUSTOMERS,
ROUND(AVG(Sales), 2) AS AVG_ORDER_VALUE
FROM superstore
""","Overall Business Performance")


  Overall Business Performance


,TOTAL_REVENUE,TOTAL_PROFIT,PROFIT_MARGIN_PCT,TOTAL_ORDERS,TOTAL_CUSTOMERS,AVG_ORDER_VALUE
0,2297200.86,286397.02,12.03,5009,793,229.86


* Store generated USD 2.3 million in revenue.
* Only USD 2 profit margin
* For every USD100 sold only USD 12 is profit
* 793 customers placed 5009 orders
* Each customers placed average 6 orders
* Average order value USD 229.86

In [9]:
 run_query("""
 SELECT 
 DISTINCT Region,
 ROUND(SUM(Sales),2) as Total_sales,
 ROUND(SUM(Profit), 2) as Total_profit,
 ROUND(AVG(Profit/Sales)*100, 2) as Profit_Margin_PCT,
 COUNT(DISTINCT "Order ID") as Total_Orders
 FROM superstore
 GROUP BY Region
 ORDER BY Total_sales DESC
 """,'Sales By Region')


  Sales By Region


,Region,Total_sales,Total_profit,Profit_Margin_PCT,Total_Orders
0,West,725457.82,108418.45,21.95,1611
1,East,678781.24,91522.78,16.72,1401
2,Central,501239.89,39706.36,-10.41,1175
3,South,391721.91,46749.43,16.35,822


"Central region has negative profit margin 
of -10.41% despite $500K in sales — 
this is a major loss area that needs 
immediate investigation!"


* West region is most profitable with profit margin of 21.95%
* Central region is making loss with negative profit margin of -10.41%
* West region has the highest sales of USD 725457.82
* West region has most number of orders 1611



In [10]:
run_query("""
SELECT 
DISTINCT Category,
ROUND(SUM(Sales), 2) as Total_Sales,
ROUND(SUM(PROFIT), 2) as Total_Profit,
ROUND(AVG(Profit/Sales)*100, 2) as Profit_Margin_PCT,
ROUND(AVG(Discount)*100, 2) as Avg_Discount_PCT,
COUNT(DISTINCT "Order Id") as Total_Orders
FROM superstore
GROUP BY Category
ORDER BY Total_sales DESC
""", "Sales By Category")


  Sales By Category


,Category,Total_Sales,Total_Profit,Profit_Margin_PCT,Avg_Discount_PCT,Total_Orders
0,Technology,836154.03,145454.95,15.61,13.23,1544
1,Furniture,741999.80,18451.27,3.88,17.39,1764
2,Office Supplies,719047.03,122490.80,13.80,15.73,3742


"Technology is the highest profitable category with total sales of $836154.03 and profit margin of 15.61%."
 Office Supplies have most orders - 3742

* Category     Discount    Margin
* Technology →  13.23%  →  15.61%
* Office Sup →  15.73%  →  13.80%
* Furniture  →  17.39%  →   3.88% 

Higher discount = Lower profit margin!
This proves discounts are hurting Furniture!

In [11]:
run_query("""
SELECT
DISTINCT "Sub-Category",
ROUND(SUM(Sales), 2) as Total_Sales,
ROUND(SUM(PROFIT), 2) as Total_Profit,
ROUND(AVG(Profit/Sales)*100, 2) as Profit_Margin_PCT,
ROUND(AVG(Discount)*100, 2) as Avg_Discount_PCT,
COUNT(DISTINCT "Order Id") as Total_Orders
FROM superstore
GROUP BY "Sub-Category"
ORDER BY Profit_Margin_PCT DESC
""", "Sales By Sub-Category")



  Sales By Sub-Category


,Sub-Category,Total_Sales,Total_Profit,Profit_Margin_PCT,Avg_Discount_PCT,Total_Orders
0,Labels,12486.31,5546.25,42.97,6.87,346
1,Paper,78479.21,34053.57,42.56,7.49,1191
2,Envelopes,16476.40,6964.18,42.31,8.03,249
3,Copiers,149528.03,55617.82,31.72,16.18,68
4,Fasteners,3024.28,949.52,29.92,8.20,215
5,Art,27118.79,6527.79,25.16,7.49,731
6,Accessories,167380.32,41936.64,21.82,7.85,718
7,Furnishings,91705.16,13059.14,13.71,13.83,877
8,Phones,330007.05,44515.73,11.92,15.46,814
9,Supplies,46673.54,-1189.10,11.20,7.68,187



* Sub-Category  |  Discount  |  Margin
* Labels       →   6.87%  →  42.97%
* Paper        →   7.49%  →  42.56%
* Binders      →  37.23%  → -19.96%
* Tables       →  26.13%  → -14.77%
* Machines     →  30.61%  →  -7.20% 


Pattern is clear:
High discount = Negative profit!

* Top products by sales - Phones, Chairs, Storage
* Top products by profit - Labels, Paper, Envelopes
* High sales but negative profit - Tables, Bookcases, Binders
* Which sub-category sells most - Binders (1316 orders)
* Which sub-category losing money- Tables, Bookcases, Binders, Machines
* Which category gets highest discount - Binders 37.23%
* At what discount profit goes negative - Above 20% discount

In [12]:
run_query("""
    SELECT 
        "Product Name",
        ROUND(SUM(Sales), 2)                 as Total_Sales,
        ROUND(SUM(Profit), 2)                as Total_Profit,
        ROUND(SUM(Profit)/SUM(Sales)*100, 2) as Profit_Margin_Pct,
        ROUND(AVG(Discount)*100, 2)          as Avg_Discount_Pct
    FROM superstore
    GROUP BY "Product Name"
    ORDER BY Total_Sales DESC
    LIMIT 10
""", "Top 10 Products by Sales")


  Top 10 Products by Sales


,Product Name,Total_Sales,Total_Profit,Profit_Margin_Pct,Avg_Discount_Pct
0,Canon imageCLASS 2200 Advanced Copier,61599.82,25199.93,40.91,12.00
1,Fellowes PB500 Electric Punch Plastic Comb Bin...,27453.38,7753.04,28.24,24.00
2,Cisco TelePresence System EX90 Videoconferenci...,22638.48,-1811.08,-8.00,50.00
3,HON 5400 Series Task Chairs for Big and Tall,21870.58,0.00,0.00,20.00
4,GBC DocuBind TL300 Electric Binding System,19823.48,2233.51,11.27,30.00
5,GBC Ibimaster 500 Manual ProClick Binding System,19024.50,760.98,4.00,52.22
6,Hewlett Packard LaserJet 3310 Copier,18839.69,6983.88,37.07,20.00
7,HP Designjet T520 Inkjet Large Format Printer ...,18374.90,4094.98,22.29,16.67
8,GBC DocuBind P400 Electric Binding System,17965.07,-1878.17,-10.45,45.00
9,High Speed Automatic Electric Letter Opener,17030.31,-262.00,-1.54,6.67


Low Discount  → High Profit  
Canon  → 12% discount → 40.91% margin

High Discount → Loss ❌
Cisco  → 50% discount → -8% margin
GBC    → 45% discount → -10.45% margin

In [13]:
run_query("""
    SELECT 
        "Product Name",
        ROUND(SUM(Sales), 2)                 as Total_Sales,
        ROUND(SUM(Profit), 2)                as Total_Profit,
        ROUND(SUM(Profit)/SUM(Sales)*100, 2) as Profit_Margin_Pct,
        ROUND(AVG(Discount)*100, 2)          as Avg_Discount_Pct
    FROM superstore
    GROUP BY "Product Name"
    ORDER BY Total_Profit DESC
    LIMIT 10
""", "Top 10 Products by Profit")


  Top 10 Products by Profit


,Product Name,Total_Sales,Total_Profit,Profit_Margin_Pct,Avg_Discount_Pct
0,Canon imageCLASS 2200 Advanced Copier,61599.82,25199.93,40.91,12.00
1,Fellowes PB500 Electric Punch Plastic Comb Bin...,27453.38,7753.04,28.24,24.00
2,Hewlett Packard LaserJet 3310 Copier,18839.69,6983.88,37.07,20.00
3,Canon PC1060 Personal Laser Copier,11619.83,4570.93,39.34,15.00
4,HP Designjet T520 Inkjet Large Format Printer ...,18374.90,4094.98,22.29,16.67
5,Ativa V4110MDD Micro-Cut Shredder,7699.89,3772.95,49.00,0.00
6,"3D Systems Cube Printer, 2nd Generation, Magenta",14299.89,3717.97,26.00,0.00
7,Plantronics Savi W720 Multi-Device Wireless He...,9367.29,3696.28,39.46,5.71
8,Ibico EPK-21 Electric Binding System,15875.92,3345.28,21.07,33.33
9,Zebra ZM400 Thermal Label Printer,6965.70,3343.54,48.00,0.00



* 0% discount   → 26-49% margin
* 5% discount   → 39% margin
* 12% discount  → 40% margin
* 33% discount  → 21% margin
* 50% discount  → -8% margin 
  

In [14]:
run_query("""
   SELECT 
       "Product Name",
       ROUND(SUM(Sales), 2) as Total_Sales,
       ROUND(SUM(Profit), 2 ) as Total_Profit,
       ROUND(SUM(Profit/Sales) * 100, 2) as Profit_Margin_Pct,
       ROUND(AVG(Discount)*100, 2) as Discount_Margin_Pct
    FROM superstore
    GROUP BY "Product Name"
    HAVING Total_Profit < 0
    ORDER BY Total_Profit ASC
    LIMIT 10
""", "Top 10 Loss Making Products")
  


  Top 10 Loss Making Products


,Product Name,Total_Sales,Total_Profit,Profit_Margin_Pct,Discount_Margin_Pct
0,Cubify CubeX 3D Printer Double Head Print,11099.96,-8879.97,-285.83,53.33
1,Lexmark MX611dhe Monochrome Laser Printer,16829.90,-4589.97,-144.44,40.00
2,Cubify CubeX 3D Printer Triple Head Print,7999.98,-3839.99,-48.00,50.00
3,Chromcraft Bull-Nose Wood Oval Conference Tabl...,9917.64,-2876.12,-123.50,28.00
4,Bush Advantage Collection Racetrack Conference...,9544.73,-1934.40,-222.23,35.00
5,GBC DocuBind P400 Electric Binding System,17965.07,-1878.17,-309.00,45.00
6,Cisco TelePresence System EX90 Videoconferenci...,22638.48,-1811.08,-8.00,50.00
7,Martin Yale Chadless Opener Electric Letter Op...,16656.20,-1299.18,-61.50,10.00
8,Balt Solid Wood Round Tables,6518.75,-1201.06,-85.33,20.00
9,BoxOffice By Design Rectangular and Half-Moon ...,1706.25,-1148.44,-206.18,48.33


The top loss making products reveal two clear patterns. First excessive discounting is the primary culprit — Cubify CubeX 3D. Printers with 50-53% discounts are losing USD 12,719 combined with margins as low as -285% meaning for every USD 100 sold the store loses $285. Second conference tables as a category are consistently losing money 
across multiple products suggesting a fundamental pricing problem beyond just discounts. The only exception is Martin Yale Letter Opener which loses money despite only 10% discount indicating wrong cost pricing. Immediate action should be taken to stop discounting on 3D Printers and review conference table pricing strategy completely.

In [ ]:
run_query("""
    SELECT 
        Segment,
        COUNT(DISTINCT "Customer ID")        as Total_Customers,
        COUNT(DISTINCT "Order ID")           as Total_Orders,
        ROUND(SUM(Sales), 2)                 as Total_Sales,
        ROUND(SUM(Profit), 2)                as Total_Profit,
        ROUND(SUM(Profit)/SUM(Sales)*100, 2) as Profit_Margin_Pct,
        ROUND(AVG(Sales), 2)                 as Avg_Order_Value
    FROM superstore
    GROUP BY Segment
    ORDER BY Total_Sales DESC
""", "Customer Segment Analysis")

"Although Consumer segment generates highest 
revenue at USD 1.16M it has the lowest profit 
margin of 11.55%. Home Office segment despite 
being smallest has highest margin of 14.03% 
and highest average order value of USD 240"

In [ ]:
run_query("""
    SELECT
       Year,
       Month,
       Month_Name,
       ROUND(SUM(Sales), 2) As Total_Sales,
       ROUND(SUM(Profit), 2) As Total_Profit,
       COUNT(DISTINCT "Order ID") As Total_Orders 
    FROM superstore
    GROUP BY Year,Month, Month_Name
    ORDER BY Year, Month
    """, "Monthly Sales Trend")

* Every year follows this pattern:

* Q1 (Jan-Mar) → Low sales
* Q2 (Apr-Jun) → Medium sales
* Q3 (Jul-Sep) → September SPIKE! 
* Q4 (Oct-Dec) → November PEAK!


Sales show a clear seasonal pattern across all 4 years with January and February being consistently weakest months and September 
and November being consistently strongest. The business grew from **USD 484K in 2014 to USD 733K in 2017** showing healthy **51%** growth over 4 years. **November 2017 was the single best month with USD 118,447 in sales**. Only 2 months in 4 years showed negative profit indicating overall business health despite product level losses identified earlier.

In [15]:
run_query("""
    SELECT 
        Year,
        ROUND(SUM(Sales), 2)                 as Total_Sales,
        ROUND(SUM(Profit), 2)                as Total_Profit,
        ROUND(SUM(Profit)/SUM(Sales)*100, 2) as Profit_Margin_Pct,
        COUNT(DISTINCT "Order ID")           as Total_Orders,
        COUNT(DISTINCT "Customer ID")        as Total_Customers
    FROM superstore
    GROUP BY Year
    ORDER BY Year
""", "Yearly Sales Summary")


  Yearly Sales Summary


,Year,Total_Sales,Total_Profit,Profit_Margin_Pct,Total_Orders,Total_Customers
0,2014,484247.50,49543.97,10.23,969,595
1,2015,470532.51,61618.60,13.10,1038,573
2,2016,609205.60,81795.17,13.43,1315,638
3,2017,733215.26,93439.27,12.74,1687,693


The business shows strong and consistent growth from 2014 to 2017 with total sales growing 51% from  **USD 484K to USD 733K** and profit growing even faster at 89% from **USD 49K to USD 93K**. The only dip was in **2015** where sales slightly decreased but profit margin actually improved to 13.10% suggesting better operational efficiency. Customer base grew modestly by 16% but orders grew 74% meaning existing customers are buying more frequently — a very healthy sign for the business.

In [17]:
run_query("""
    SELECT 
        CASE 
            WHEN Discount = 0 THEN '0% No Discount'
            WHEN Discount <= 0.10 THEN '1-10% Low'
            WHEN Discount <= 0.20 THEN '11-20% Medium'
            WHEN Discount <= 0.40 THEN '21-40% High'
            ELSE '40%+ Very High'
        END as Discount_Range,
        COUNT(*) as Total_Orders,
        ROUND(AVG(Sales), 2) as Avg_Sales,
        ROUND(SUM(Sales), 2) as Total_Sales,
        ROUND(SUM(Profit), 2) as Total_Profit,
        ROUND(AVG(Profit), 2) as Avg_Profit
    FROM superstore
    GROUP BY Discount_Range
    ORDER BY Total_Profit DESC
""", "Discount Impact Analysis")


  Discount Impact Analysis


,Discount_Range,Total_Orders,Avg_Sales,Total_Sales,Total_Profit,Avg_Profit
0,0% No Discount,4798,226.74,1087908.47,320987.60,66.90
1,11-20% Medium,3709,213.58,792152.89,91756.30,24.74
2,1-10% Low,94,578.40,54369.35,9029.18,96.06
3,21-40% High,460,509.00,234137.90,-35817.47,-77.86
4,40%+ Very High,933,137.87,128632.25,-99558.59,-106.71


Discount analysis reveals a devastating truth about the store's pricing strategy. Orders with **no discount generate USD 320,987 in profit** which actually exceeds the store's total profit of USD 286,397 meaning discounted orders collectively destroy USD 135,375 of profit. Orders with **40%+ discount lose USD 106 per order on average** despite 933 such orders while no discount orders make USD 67 profit per order. Most surprisingly high discounts don't even attract bigger orders — **very high discount orders average only USD 137 per order versus USD 226 for no discount orders**. The discounting strategy is clearly destroying profitability with zero benefit to sales volume.

In [18]:
run_query("""
    SELECT 
        "Ship Mode",
        COUNT(DISTINCT "Order ID")           as Total_Orders,
        ROUND(SUM(Sales), 2)                 as Total_Sales,
        ROUND(SUM(Profit), 2)                as Total_Profit,
        ROUND(SUM(Profit)/SUM(Sales)*100, 2) as Profit_Margin_Pct,
        ROUND(AVG(Sales), 2)                 as Avg_Order_Value
    FROM superstore
    GROUP BY "Ship Mode"
    ORDER BY Total_Orders DESC
""", "Ship Mode Analysis")


  Ship Mode Analysis


,Ship Mode,Total_Orders,Total_Sales,Total_Profit,Profit_Margin_Pct,Avg_Order_Value
0,Standard Class,2994,1358215.74,164088.79,12.08,227.58
1,Second Class,964,459193.57,57446.64,12.51,236.09
2,First Class,787,351428.42,48969.84,13.93,228.50
3,Same Day,264,128363.13,15891.76,12.38,236.40


Standard Class is overwhelmingly the most popular shipping method with 2994 orders representing 60% of all orders and 
generating USD 164K in profit. However First Class shipping has the highest profit margin at 13.93% suggesting that customers who pay for faster shipping tend to purchase higher value items with better margins. All shipping modes maintain similar profit margins between 12-14% indicating shipping mode itself does not significantly impact profitability.

In [19]:
run_query("""
    SELECT 
        State,
        ROUND(SUM(Sales), 2)                 as Total_Sales,
        ROUND(SUM(Profit), 2)                as Total_Profit,
        ROUND(SUM(Profit)/SUM(Sales)*100, 2) as Profit_Margin_Pct,
        COUNT(DISTINCT "Order ID")           as Total_Orders
    FROM superstore
    GROUP BY State
    ORDER BY Total_Sales DESC
    LIMIT 10
""", "Top 10 States by Sales")


  Top 10 States by Sales


,State,Total_Sales,Total_Profit,Profit_Margin_Pct,Total_Orders
0,California,457687.63,76381.39,16.69,1021
1,New York,310876.27,74038.55,23.82,562
2,Texas,170188.05,-25729.36,-15.12,487
3,Washington,138641.27,33402.65,24.09,256
4,Pennsylvania,116511.91,-15559.96,-13.35,288
5,Florida,89473.71,-3399.30,-3.80,200
6,Illinois,80166.10,-12607.89,-15.73,276
7,Ohio,78258.14,-16971.38,-21.69,236
8,Michigan,76269.61,24463.19,32.07,117
9,Virginia,70636.72,18597.95,26.33,115


* 5 out of top 10 states are losing money!
* Texas has 3rd highest sales but is 
  2nd biggest loss maker
* High sales does not guarantee profit
* Michigan makes USD 24K profit from 
  just 117 orders vs Texas losing 
  USD 25K from 487 orders
* Quality of orders matters more 
  than quantity!

In [20]:
run_query("""
    SELECT 
        City,
        State,
        ROUND(SUM(Sales), 2)                 as Total_Sales,
        ROUND(SUM(Profit), 2)                as Total_Profit,
        ROUND(SUM(Profit)/SUM(Sales)*100, 2) as Profit_Margin_Pct,
        COUNT(DISTINCT "Order ID")           as Total_Orders
    FROM superstore
    GROUP BY City, State
    ORDER BY Total_Profit DESC
    LIMIT 10
""", "Top 10 Cities by Profit")


  Top 10 Cities by Profit


,City,State,Total_Sales,Total_Profit,Profit_Margin_Pct,Total_Orders
0,New York City,New York,256368.16,62036.98,24.20,450
1,Los Angeles,California,175851.34,30440.76,17.31,384
2,Seattle,Washington,119540.74,29156.10,24.39,212
3,San Francisco,California,112669.09,17507.39,15.54,265
4,Detroit,Michigan,42446.94,13181.79,31.05,53
5,Lafayette,Indiana,19630.45,8976.10,45.73,7
6,Newark,Delaware,20448.05,8086.17,39.54,29
7,Atlanta,Georgia,17197.84,6993.66,40.67,17
8,Minneapolis,Minnesota,16870.54,6824.58,40.45,13
9,San Diego,California,47521.03,6377.20,13.42,88


* Small cities outperform big cities 
  in profit margin
* Lafayette 45.73% margin from 7 orders
* Atlanta 40.67% margin from 17 orders
* Minneapolis 40.45% margin from 13 orders
* Quality beats quantity every time!
* NYC generates most profit but not 
  highest margin
* Detroit makes USD 13K from just 53 orders
  vs San Diego making USD 6K from 88 orders

## 🎯 Key Business Insights & Conclusions

### Overall Performance
* Store generated **$2.3 Million** in revenue with **12.03%** profit margin from 2014-2018
* Business grew **51%** in sales and **89%** in profit over 4 years
* Total of **793 customers** placed **5009 orders**

### Regional Performance

* **West region** is top performer with USD 725K sales and 21.95% margin
* **Central region** is biggest concern with -10.41% profit margin despite $500K sales


### Category Performance
* **Technology** is most profitable at 15.61% margin
* **Furniture** is least profitable at only 3.88% margin due to high discounts

### Discount Impact — Most Critical Finding
- Orders with **0% discount** generate USD 320,987 profit
- Orders with 40%+ discount generate -$99,558 loss
- High discounts are destroying USD 135,375 in profit
- **Recommendation: Stop all discounts above 20%**

### Loss Making Products
- **Cubify 3D Printers** losing $12,719 combined
  due to 50%+ discounts
- **Conference Tables** consistently losing money
  suggesting wrong pricing strategy

### Customer Segments
- **Consumer** segment has highest sales but 
  lowest margin 11.55%
- **Home Office** segment has highest margin 
  14.03% and highest avg order value $240.97

### Seasonal Trends
- **November** is consistently best month 
  every year
- **January** is consistently worst month
- Clear Q4 seasonal pattern every year

### Geographic Performance
- **5 out of top 10 states** are losing money
- **Texas** has $170K sales but -$25K loss
- **New York City** is most profitable city 
  with $62K profit

### Top 3 Recommendations
1. **Stop discounts above 20%** — saves $135K loss
2. **Focus on Home Office segment** — highest margin
3. **Review Texas and Central region** strategy — 
   both losing money despite high sales